In [1]:
import pandas as pd

# Load the training and test datasets
train_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/train.csv')
test_df = pd.read_csv('D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/04_diabetes/test.csv')

# Drop duplicate rows from the training and test datasets
train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-10 08:03:10.159 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['gender', 'smoking_history'], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import OneHotEncode

# Initialize the OneHotEncode tool with the specified categorical columns
encoder = OneHotEncode(features=['gender', 'smoking_history'])

# Fit and transform the training data
train_df_encoded = encoder.fit_transform(train_df.copy())

# Transform the test data using the same encoder
test_df_encoded = encoder.transform(test_df.copy())


D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the encoded train DataFrame
column_info_train = get_column_info(train_df_encoded)
print("Column information for train_df_encoded")
print(column_info_train)

# Check column information for the encoded test DataFrame
column_info_test = get_column_info(test_df_encoded)
print("Column information for test_df_encoded")
print(column_info_test)


Column information for train_df_encoded
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}
Column information for test_df_encoded
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# Assuming the label column is 'diabetes'
label_col = 'diabetes'

# Separate features and target variable
X_train = train_df_encoded.drop(columns=[label_col])
y_train = train_df_encoded[label_col]
X_test = test_df_encoded.drop(columns=[label_col])
y_test = test_df_encoded[label_col]

# Build a random forest classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)

# Rank the most important features using permutation importance
result = permutation_importance(rf_classifier, X_train, y_train, n_repeats=10, random_state=42, n_jobs=-1)

# Create a DataFrame to display feature importances
feature_importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean
})

# Sort features by importance
feature_importances = feature_importances.sort_values(by='importance', ascending=False)

# Display the most important features
print(feature_importances)


                        feature  importance
4                   HbA1c_level    0.082175
5           blood_glucose_level    0.070502
0                           age    0.027748
3                           bmi    0.026443
7                   gender_Male    0.010715
9       smoking_history_No Info    0.010181
13        smoking_history_never    0.009472
6                 gender_Female    0.008675
1                  hypertension    0.007948
12       smoking_history_former    0.006293
2                 heart_disease    0.005114
10      smoking_history_current    0.003446
14  smoking_history_not current    0.002898
11         smoking_history_ever    0.001887
8                  gender_Other    0.000000


In [6]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Check column information for the encoded training dataframe
column_info_train = get_column_info(train_df_encoded)
print("Column information for train_df_encoded")
print(column_info_train)

# Check column information for the encoded test dataframe
column_info_test = get_column_info(test_df_encoded)
print("Column information for test_df_encoded")
print(column_info_test)


Column information for train_df_encoded
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}
Column information for test_df_encoded
{'Category': [], 'Numeric': ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current'], 'Datetime': [], 'Others': []}


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

# Define the label column
label_col = 'diabetes'

# Split the data into features and target
X_train = train_df_encoded.drop(columns=[label_col])
y_train = train_df_encoded[label_col]
X_test = test_df_encoded.drop(columns=[label_col])
y_test = test_df_encoded[label_col]

# Initialize and train the RandomForestClassifier
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area under ROC curve: {roc_auc:.4f}')

# Show the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(conf_matrix)


Area under ROC curve: 0.9605
Confusion Matrix:
[[18035    64]
 [  533  1175]]


In [8]:
# Make predictions on the test set using the trained model
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Display the predictions
print("Predictions on the test set:")
print(y_pred)


Predictions on the test set:
[0 0 0 ... 0 0 0]


In [9]:
from sklearn.metrics import roc_auc_score

# Ensure the test data is processed the same way as the training data
X_test = test_df_encoded.drop(columns=[label_col])
y_test = test_df_encoded[label_col]

# Use the trained model to predict probabilities on the test set
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'Area under ROC curve: {roc_auc:.4f}')


Area under ROC curve: 0.9605


In [10]:
from sklearn.metrics import confusion_matrix

# Ensure the test data is processed the same way as the training data
X_test = test_df_encoded.drop(columns=[label_col])
y_test = test_df_encoded[label_col]

# Use the trained model to make predictions
y_pred = rf_classifier.predict(X_test)

# Compute the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
print('Confusion Matrix:')
print(conf_matrix)


Confusion Matrix:
[[18035    64]
 [  533  1175]]
